# CEH GEAR Monthly Rainfall: NetCDF via Python + Xarray

**Launch this notebook:**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_ORG/YOUR_REPO/blob/main/gear_netcdf_python.ipynb)
[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/YOUR_ORG/YOUR_REPO/HEAD?labpath=gear_netcdf_python.ipynb)
[![Open in DataLabs](https://img.shields.io/badge/Open%20in-NERC%20DataLabs-blue?logo=jupyter)](https://datalabs.nerc.ac.uk)

> Replace `YOUR_ORG/YOUR_REPO` with your GitHub path once published.

---

## What this notebook does

This notebook explores the **CEH GEAR monthly rainfall** dataset — 1 km gridded estimates of monthly rainfall for Great Britain, stored in NetCDF format. We use **xarray**, which reads NetCDF natively and gives you a clean, labelled view of the data.

**We will:**
1. Download and open a NetCDF file
2. Inspect its metadata (dimensions, variables, attributes)
3. Extract and plot a time series at a single location
4. Plot a map of rainfall for a single month

**Data:** 1 km monthly rainfall, Great Britain, 1890–2019  
**Source:** CEH-GEAR via EIDC  
**CRS:** British National Grid (EPSG:27700) — coordinates are easting/northing in metres

**Libraries used:** `xarray`, `matplotlib`, `numpy` — all standard and pre-installed on Colab.

| Runtime | Notes |
|---------|-------|
| Google Colab | ✅ Works out of the box |
| Binder | ✅ Works out of the box |
| NERC DataLabs | ✅ Full support |
| JupyterLite | ✅ Should work; download cell uses `wget` — see note below |
| Local Jupyter | ✅ `pip install xarray matplotlib` if needed |
| Quarto | ✅ Use `jupyter` engine |

> **JupyterLite note:** If `wget` is not available, download the file manually from the URL below and upload it to your session, then skip the download cell.

---

## 0. Setup

In [ ]:
%pip install -q xarray matplotlib numpy

In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt

print(f"xarray version: {xr.__version__}")

---
## 1. Download the NetCDF file

We download a single year (1894) of monthly GEAR data.  
Each file contains 12 monthly time steps for the full 1 km GB grid.

> **Note:** The file is ~60 MB. The download takes ~30 seconds on Colab.

In [ ]:
nc_url = (
    "https://catalogue.ceh.ac.uk/datastore/eidchub/"
    "dbf13dd5-90cd-457a-a986-f2f9dd97e93c/GB/monthly/CEH_GEAR_monthly_GB_1894.nc"
)
nc_file = "CEH_GEAR_monthly_GB_1894.nc"

# Download using wget (works on Colab, Binder, DataLabs, local Linux/Mac)
!wget -q --show-progress -O {nc_file} "{nc_url}"
print(f"\nSaved to: {nc_file}")

---
## 2. Open the file and inspect metadata

In [ ]:
ds = xr.open_dataset(nc_file)

# Display the dataset summary — dimensions, variables, coordinates, attributes
ds

### 2a. Global attributes

In [ ]:
print("=== Global attributes ===")
for name, value in ds.attrs.items():
    val_str = str(value)
    if len(val_str) > 100:
        val_str = val_str[:97] + "..."
    print(f"  {name:<30} : {val_str}")

### 2b. Dimensions, coordinates and variables

In [ ]:
print("=== Dimensions ===")
for dim, size in ds.dims.items():
    print(f"  {dim:<10} : {size} values")

print("\n=== Coordinates ===")
for name, coord in ds.coords.items():
    vals = coord.values
    units = coord.attrs.get("units", "")
    if vals.ndim == 1:
        print(f"  {name:<10} : {vals.min():.2f} to {vals.max():.2f}  [{units}]  ({len(vals)} values)")
    else:
        print(f"  {name:<10} : shape {vals.shape}  [{units}]")

print("\n=== Data variables ===")
for name, var in ds.data_vars.items():
    long_name = var.attrs.get("long_name", "")
    units = var.attrs.get("units", "")
    print(f"  {name:<30} {var.dims}  [{units}]  — {long_name}")

### 2c. Coordinate system note

The GEAR NetCDF uses **British National Grid (BNG, EPSG:27700)** — coordinates are easting (`x`) and northing (`y`) in metres, not longitude/latitude. The file also contains `lat` and `lon` 2-D arrays that give the geographic equivalent for each grid cell.

In [ ]:
# The x/y dimensions are BNG easting/northing in metres
x = ds["x"].values   # easting, metres
y = ds["y"].values   # northing, metres

print(f"Easting  (x): {x.min():,.0f} m to {x.max():,.0f} m  ({len(x)} values, {x[1]-x[0]:.0f} m spacing)")
print(f"Northing (y): {y.min():,.0f} m to {y.max():,.0f} m  ({len(y)} values, {abs(y[1]-y[0]):.0f} m spacing)")
print(f"\nTime steps: {len(ds['time'])}")
print(f"Time range: {str(ds['time'].values[0])[:10]} to {str(ds['time'].values[-1])[:10]}")

---
## 3. Time series at a single location

The grid uses BNG easting/northing. We select by those coordinates.  
If you have a lat/lon, convert it first (e.g. using `pyproj`).

In [ ]:
# Edinburgh in British National Grid (easting, northing in metres)
# BNG: Edinburgh ~ (325000, 673000)
# You can convert lon/lat to BNG at: https://gridreferencefinder.com
target_x = 325000   # easting in metres
target_y = 673000   # northing in metres

# Select nearest grid cell using BNG coordinates
point_ts = ds["rainfall_amount"].sel(x=target_x, y=target_y, method="nearest")

actual_x = float(point_ts.x)
actual_y = float(point_ts.y)
print(f"Target  : x={target_x:,.0f}, y={target_y:,.0f}")
print(f"Nearest : x={actual_x:,.0f}, y={actual_y:,.0f}")
print(f"Values  : {point_ts.values}")

In [ ]:
units = ds["rainfall_amount"].attrs.get("units", "mm")
long_name = ds["rainfall_amount"].attrs.get("long_name", "Rainfall amount")

fig, ax = plt.subplots(figsize=(10, 4))

ax.bar(range(len(point_ts)), point_ts.values, color="#1d6fa4", alpha=0.85)
ax.set_xticks(range(len(point_ts)))
ax.set_xticklabels(
    [str(t)[:7] for t in ds["time"].values],   # show YYYY-MM
    rotation=45, ha="right"
)
ax.set_title(f"CEH GEAR Monthly: {long_name}", fontsize=13, fontweight="bold")
ax.set_ylabel(f"{long_name} ({units})")
ax.text(0.01, 0.97, f"BNG: x={actual_x:,.0f} m, y={actual_y:,.0f} m",
        transform=ax.transAxes, va="top", fontsize=9, color="grey")
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
plt.show()

---
## 4. Map at a single time step

Select one month and plot the full GB grid. The x-axis is easting and y-axis is northing (in metres), so the map will be in BNG projection — which looks correct for GB.

In [ ]:
# Select the first time step (January 1894)
t_index = 0
slice_2d = ds["rainfall_amount"].isel(time=t_index)
time_label = str(ds["time"].values[t_index])[:7]

print(f"Plotting: {time_label}")
print(f"Grid shape: {slice_2d.shape}")
print(f"Value range: {float(slice_2d.min()):.1f} to {float(slice_2d.max()):.1f} {units}")

fig, ax = plt.subplots(figsize=(6, 9))

# xarray .plot() automatically uses x and y dimensions as axes
slice_2d.plot(
    ax=ax,
    cmap="YlGnBu",
    cbar_kwargs={"label": f"{long_name} ({units})", "shrink": 0.6}
)

# Mark the selected point
ax.plot(actual_x, actual_y, "r+", markersize=10, markeredgewidth=2, label="Selected point")
ax.legend(fontsize=9)

ax.set_title(f"CEH GEAR Monthly: {long_name}\n{time_label}", fontsize=12, fontweight="bold")
ax.set_xlabel("Easting (m, BNG)")
ax.set_ylabel("Northing (m, BNG)")
ax.set_aspect("equal")
fig.tight_layout()
plt.show()

---
## 5. Annual total map

Sum all 12 months to get the annual total rainfall.

In [ ]:
annual = ds["rainfall_amount"].sum(dim="time")

fig, ax = plt.subplots(figsize=(6, 9))

annual.plot(
    ax=ax,
    cmap="YlGnBu",
    cbar_kwargs={"label": f"Annual total ({units})", "shrink": 0.6}
)

year = str(ds["time"].values[0])[:4]
ax.set_title(f"CEH GEAR: Annual total rainfall {year}", fontsize=12, fontweight="bold")
ax.set_xlabel("Easting (m, BNG)")
ax.set_ylabel("Northing (m, BNG)")
ax.set_aspect("equal")
fig.tight_layout()
plt.show()

---
## Tips

| Task | Code |
|------|------|
| Select a specific month | `ds["rainfall_amount"].sel(time="1894-06")` |
| Spatial mean (area-averaged) | `ds["rainfall_amount"].mean(dim=["x", "y"])` |
| Convert BNG to lon/lat | Use the 2-D `lat`/`lon` arrays already in the file |
| Load another year | Change the URL year (`_1894_` → `_1900_` etc.) |
| Save processed data | `ds.to_netcdf("output.nc")` |

**Data citation:** CEH-GEAR, UKCEH. https://doi.org/10.5285/dbf13dd5-90cd-457a-a986-f2f9dd97e93c